<a href="https://colab.research.google.com/github/yccccc12/DocStruct-VLM/blob/master/experiments/DeepSeekOCR_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/deepseek-ai/DeepSeek-OCR-2.git
%cd DeepSeek-OCR-2
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0
!pip install -r requirements.txt
!pip install flash-attn==2.7.3 --no-build-isolation


Cloning into 'DeepSeek-OCR-2'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 67 (delta 10), reused 5 (delta 4), pack-reused 47 (from 1)
Receiving objects: 100% (67/67), 1.06 MiB | 4.05 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/DeepSeek-OCR-2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 93.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
  

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch, os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
model_name = "deepseek-ai/DeepSeek-OCR-2"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
)

model = model.eval()

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR2. This is not supported for all configurations of models and can yield errors.


In [ ]:
prompt = "<image>\n<|grounding|>Convert the image to markdown. "
image_file = "/content/Test1.png"
output_path = "output.md"
res = model.infer(tokenizer, prompt=prompt, image_file=image_file, output_path=output_path, base_size=1024, image_size=768, crop_mode=True, save_results=True)

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([4, 144, 1280])
<|ref|>sub_title<|/ref|><|det|>[[12, 12, 200, 45]]<|/det|>
## 3 Experiments

<|ref|>text<|/ref|><|det|>[[9, 80, 985, 141]]<|/det|>
In this section, we first introduce the overall model and compare it with the current state-of-the-art (SoTA) models. Then, we evaluate the model's performance across various sub-capabilities.

<|ref|>sub_title<|/ref|><|det|>[[10, 176, 411, 207]]<|/det|>
## 3.1 Comparison with the SOTA Models

<|ref|>figure_title<|/ref|><|det|>[[216, 243, 775, 275]]<|/det|>
Table 3: Performance of Qwen2.5-VL and State-of-the-art.

<|ref|>table<|/ref|><|det|>[[20, 295, 974, 792]]<|/det|>
<table><tr><td rowspan="2">Datasets</td><td>Previous</td><td>Claude-3.5</td><td>GPT-4o</td><td>InternVL2.5</td><td>Qwen2-VL</td><td>Qwen2.5-VL</td><td>Qwen2.5-VL</td><td>Qwen2.5-V</td></tr><tr><td>Open-source SoTA</td><td>Sonnet-0620</td><td>0513</td><td>78B</td><td>72B</td><td>72B</td><td>7B</td><td>3B</td></tr><tr><td c

image: 0it [00:00, ?it/s]
other: 100%|██████████| 6/6 [00:00<00:00, 41665.27it/s]


In [ ]:
import fitz, io
from PIL import Image

def pdf_to_images(pdf_path, dpi=144):
    doc = fitz.open(pdf_path)
    zoom = dpi / 72.0
    matrix = fitz.Matrix(zoom, zoom)
    images = []
    for page in doc:
        pix = page.get_pixmap(matrix=matrix, alpha=False)
        img = Image.open(io.BytesIO(pix.tobytes("png")))
        if img.mode in ("RGBA", "LA"):
            bg = Image.new("RGB", img.size, (255, 255, 255))
            bg.paste(img, mask=img.split()[-1] if img.mode == "RGBA" else None)
            img = bg
        images.append(img)
    doc.close()
    return images

In [ ]:
pdf_path = "/content/UECM3993_MAY2025-1-3.pdf"
images = pdf_to_images(pdf_path)
prompt = "<image>\n<|grounding|>Convert the document to markdown."

for idx, img in enumerate(images):
    img_path = f"page_{idx:03d}.png"
    img.save(img_path)
    output_path = f"page_{idx:03d}.md"
    _ = model.infer(
        tokenizer,
        prompt=prompt,
        image_file=img_path,
        output_path=output_path,
        base_size=1024,
        image_size=768,
        crop_mode=True,
        save_results=True
    )

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([6, 144, 1280])
<|ref|>title<|/ref|><|det|>[[312, 62, 677, 81]]<|/det|>
# UNIVERSITI TUNKU ABDUL RAHMAN

<|ref|>title<|/ref|><|det|>[[360, 103, 626, 122]]<|/det|>
# ACADEMIC YEAR 2024/2025

<|ref|>title<|/ref|><|det|>[[371, 139, 616, 158]]<|/det|>
# MAY 2025 EXAMINATION

<|ref|>title<|/ref|><|det|>[[306, 174, 679, 193]]<|/det|>
# UECM3993 PREDICTIVE MODELLING

<|ref|>text<|/ref|><|det|>[[111, 210, 342, 228]]<|/det|>
TUESDAY, 3 JUNE 2025

<|ref|>text<|/ref|><|det|>[[526, 210, 874, 228]]<|/det|>
TIME : 9.00 AM – 11.00 AM (2 HOURS)

<|ref|>title<|/ref|><|det|>[[109, 264, 875, 317]]<|/det|>
# BACHELOR OF SCIENCE (HONOURS) ACTUARIAL SCIENCE
BACHELOR OF SCIENCE (HONOURS) APPLIED MATHEMATICS WITH COMPUTING
BACHELOR OF SCIENCE (HONOURS) FINANCIAL MATHEMATICS

<|ref|>sub_title<|/ref|><|det|>[[108, 368, 355, 386]]<|/det|>
## Instructions to Candidates :

<|ref|>text<|/ref|><|det|>[[157, 397, 469, 415]]<|/det|>
• Answer ALL questions in PART 

image: 0it [00:00, ?it/s]
other: 100%|██████████| 12/12 [00:00<00:00, 120410.64it/s]
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([6, 144, 1280])
<|ref|>sub_title<|/ref|><|det|>[[105, 74, 478, 92]]<|/det|>
## UECM3993 PREDICTIVE MODELLING

<|ref|>sub_title<|/ref|><|det|>[[105, 108, 390, 126]]<|/det|>
## PART A: Answer ALL questions.

<|ref|>text<|/ref|><|det|>[[104, 150, 869, 185]]<|/det|>
Q1. (a) Supervised learning models can be classified into parametric models and non-parametric models.

<|ref|>text<|/ref|><|det|>[[224, 190, 870, 226]]<|/det|>
(i) Explain the differences between these two classes of models with respect to the number of samples n. (3 marks)

<|ref|>text<|/ref|><|det|>[[225, 241, 870, 275]]<|/det|>
(ii) State two different classes of predictive models that are parametric. (2 marks)

<|ref|>text<|/ref|><|det|>[[225, 292, 870, 327]]<|/det|>
(iii) State two different classes of predictive models that are nonparametric. (2 marks)

<|ref|>text<|/ref|><|det|>[[165, 363, 872, 416]]<|/det|>
(b) Given a time-based data (e.g. econometric data), state

image: 0it [00:00, ?it/s]
other: 100%|██████████| 18/18 [00:00<00:00, 96915.88it/s]
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([6, 144, 1280])
<|ref|>sub_title<|/ref|><|det|>[[111, 81, 483, 99]]<|/det|>
## UECM3993 PREDICTIVE MODELLING

<|ref|>sub_title<|/ref|><|det|>[[110, 115, 323, 134]]<|/det|>
## PART A Q1. (Continued)

<|ref|>text<|/ref|><|det|>[[168, 153, 874, 189]]<|/det|>
(d) AI is adopted in various industries but Malaysia is currently dependent on AI on the cloud such as ChatGPT which are closed source software.

<|ref|>text<|/ref|><|det|>[[229, 194, 874, 230]]<|/det|>
(i) State the dangers of relying on AI on the cloud from a data security and a computer security point of view. (2 marks)

<|ref|>text<|/ref|><|det|>[[229, 245, 875, 298]]<|/det|>
(ii) With the release of open source AI software with liberal licensing such as DeepSeek V3, state the opportunities and challenges faced by Malaysia to deploy open source AI software. (3 marks)

<|ref|>text<|/ref|><|det|>[[722, 295, 874, 312]]<|/det|>
[Total : 25 marks]
===============save results:======

image: 0it [00:00, ?it/s]
other: 100%|██████████| 6/6 [00:00<00:00, 49344.75it/s]
